# Multi-Hop Retrieval — Train Model 1 (ComplementEncoder)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Your Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`
   - `musique_ans_v1.0_dev.jsonl`
4. Folder must be shared as **Anyone with the link can view**

**What this trains:**
- `ComplementEncoder` — joint BERT `[CLS]A[SEP]B[SEP]` → B-side token matrix [n×128]
- Three-component loss: `L_content + L_orthogonal + L_chain`
  - `L_content`: complement must encode B's information
  - `L_orthogonal`: complement must NOT encode A's information (prevents shortcut)
  - `L_chain`: C-anchor contrastive — complement points toward next hop C
- Dataset: all ~26K MuSiQue hop pairs

**Expected time: ~2 hours on T4**

**After training:** `model1_complement.pt` is copied to `/kaggle/working/` — download it from the **Output tab** (right panel) and upload to your Google Drive `musique_data/` folder.

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU — must be T4 (sm_70+), not P100 (sm_60)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Settings → Accelerator → T4 GPU")

props   = torch.cuda.get_device_properties(0)
cc      = props.major
vram_gb = props.total_memory / 1e9

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"SM   : {cc}.{props.minor}")

if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_{cc}0) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download MuSiQue data from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up expected file paths
import os

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Smoke test — verify data loader and dataset size
import os
os.chdir(WORK_DIR)

# Should print:
#   [ChainDataset] ~26,675 examples (6,737 with C-anchor, 19,938 without)
os.system("python data_loader.py")

---
## Train Model 1 — ComplementEncoder

**Architecture:** joint BERT `[CLS]A[SEP]B[SEP]` → B-side token matrix [n×128] L2-norm

**Loss (three components):**
- `L_content  (w=1.0)` — complement must encode B's content
- `L_orthogonal (w=0.3)` — complement must differ from A (prevents ignoring A)
- `L_chain (w=0.5)` — C-anchor: complement points toward next hop C (only when C exists)

**Dataset:** ~26,675 MuSiQue hop pairs (all hops, not just 3/4-hop)

In [ ]:
# 6. Train Model 1 (full 3-epoch run)
import os
os.chdir(WORK_DIR)

log_file = "/kaggle/working/model1_train.log"
os.system(f"python model1_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify checkpoints and copy to /kaggle/working/ root for easy download
import os, shutil

best  = f"{WORK_DIR}/models/model1_complement.pt"
final = f"{WORK_DIR}/models/model1_complement_final.pt"

for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND — check training log")

# Copy best checkpoint to /kaggle/working/ root — appears in Output tab for download
if os.path.exists(best):
    out_path = "/kaggle/working/model1_complement.pt"
    shutil.copy(best, out_path)
    print(f"\n  Copied to {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")
    print("  → Download from the Output tab (right panel) and upload to your Drive musique_data/ folder")

print("\n--- Last 30 lines of training log ---")
with open("/kaggle/working/model1_train.log") as f:
    lines = f.readlines()
print("".join(lines[-30:]))

---
## Done

`model1_complement.pt` is in `/kaggle/working/` — visible in the **Output tab** on the right.

**To save to Google Drive:**
1. Output tab → find `model1_complement.pt` → click download
2. Upload it to your Google Drive `musique_data/` folder (replace the old one)

**Next step:** Run `train_m2_kaggle.ipynb` to train Model 2 using this new Model 1 checkpoint.